In [2]:
import pandas as pd
from sklearn import model_selection
from pipeline import (
    create_encoder_pipeline,
    model_attributes,
)
from load import load_dataset

dataset = load_dataset("futbol_uruguayo.csv")

print(dataset.head())


                   home                              away       date  gh  ga  \
0           Bella Vista                 Defensor Sporting 1932-03-05   1   2   
1            CA Penarol                       River Plate 1932-03-05   1   1   
2       Central Espanol        Rampla Juniors Futbol Club 1932-03-05   1   0   
3  Montevideo Wanderers                       Racing Club 1932-03-05   3   0   
4              Nacional  Institucion Atletica Sud America 1932-03-05   2   0   

  result record last_matches goal_difference  local_experience  \
0      V      E            E               E                 0   
1      E      E            E               E                 0   
2      L      E            E               E                 0   
3      L      E            E               E                 0   
4      L      E            E               E                 0   

   away_experience  record_enough  
0                0              0  
1                0              0  
2             

In [ ]:
#separamos cronologicamente el conjunto de entrenamiento y el de evaluacion
#la evaluacion se mantiene separada hasta haber elegido los hiperparametros

train = dataset[
    dataset["date"] < pd.Timestamp("2024-01-01")
].copy()

test = dataset[
    (dataset["date"] >= pd.Timestamp("2024-01-01"))
    & (dataset["date"] < pd.Timestamp("2026-01-01"))
].copy()

print("Cantidad de partidos de entrenamiento:", len(train))
print("Cantidad de partidos de evaluacion:", len(test))

Cantidad de partidos de entrenamiento: 14734
Cantidad de partidos de evaluacion: 472


In [ ]:
from sklearn.metrics import normalized_mutual_info_score

normalized_mutual_info_score(
    train["record"],
    train["last_matches"]
)

In [ ]:
#creamos el pipeline base
base_pipeline= create_base_pipeline()

base_df_procesado= base_pipeline.fit_transform(df_completo)

print(
    base_df_procesado[
        [
            #"date",
            "home",
            "away",
            "win_rate",
            "result"
        ]
    ].head(20)
)


In [ ]:
#separamos los atributos de entrada y la clase que queremos predecir
X_train = train[model_attributes].copy()
y_train = train["result"].copy()

X_test = test[model_attributes].copy()
y_test = test["result"].copy()

#el encoder aprende la codificacion usando solamente entrenamiento
encoder_pipeline = create_encoder_pipeline()

X_train_encoded = encoder_pipeline.fit_transform(
    X_train
)

#en evaluacion reutilizamos exactamente la codificacion aprendida antes
X_test_encoded = encoder_pipeline.transform(
    X_test
)

print(X_train_encoded.head(20))

    record  last_matches  goal_difference  local_experience  away_experience  \
0        1             1                1                 0                0   
1        1             1                1                 0                0   
2        1             1                1                 0                0   
3        1             1                1                 0                0   
4        1             1                1                 0                0   
5        1             2                2                 1                1   
6        2             2                2                 1                1   
7        1             1                2                 1                1   
8        2             2                2                 1                1   
9        1             1                1                 1                1   
10       1             1                2                 1                1   
11       2             2                

In [5]:
import sys
import os

# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))


#hacemos el arbol de decision usando solamente el conjunto de entrenamiento
from decisionTree.clasifier import Clasifier as DecisionTreeClassifier

modelo = DecisionTreeClassifier(0.003)

modelo.fit(X_train_encoded, y_train)
modelo.tree.print_tree()

record
  [1]
    goal_difference
      [1]
        last_matches
          [1]
            away_experience
              [0]
                L
              [1]
                L
              [3]
                L
          [0]
            L
          [2]
            record_enough
              [1]
                E
              [0]
                V
      [2]
        record_enough
          [0]
            last_matches
              [2]
                away_experience
                  [1]
                    L
                  [2]
                    L
                  [3]
                    local_experience
                      [1]
                        L
                      [2]
                        E
              [1]
                away_experience
                  [1]
                    E
                  [2]
                    L
              [0]
                E
          [1]
            last_matches
              [2]
                L
              [1]
       